In [37]:
import os
import pandas as pd
import json
import re

# Funktion zum Extrahieren von Zahlen für die richtige Sortierung
def sort_instances(instance_name):
    numbers = re.findall(r'\d+', instance_name)
    return tuple(int(num) for num in numbers)  # Zahlen extrahieren und als Tupel zurückgeben

def shorten_instance_name(instance_name):
    # Instanzname kürzen, z.B. "a10_o107_m5_an57_ar12" → "a10_o107"
    parts = instance_name.split("_")
    return "_".join(parts[:2])

# Basispfad (aktueller Ordner des Jupyter Notebooks)
solution_path = os.getcwd()

# Benutzerdefinierte Strategiereihenfolge
strategy_order = {"costs": 0, "weighted": 1, "hierarchical": 2, "hierarchical_tolerance": 3}

# Liste für die Daten
data_objectives = []

# Ordner rekursiv durchlaufen
for root, dirs, files in os.walk(solution_path):
    for file in files:
        if file.endswith(".json") and "Solution_Construction" in file:  # Nur passende JSON-Dateien berücksichtigen
            file_path = os.path.join(root, file)
            
            # JSON-Datei einlesen mit Fehlerbehandlung
            try:
                with open(file_path, 'r') as f:
                    data = json.load(f)
            except json.JSONDecodeError:
                print(f"Ungültige JSON-Datei übersprungen: {file_path}")
                continue
            
            # Instanzname, Objective-Type und Strategie aus dem Pfad extrahieren
            instance = shorten_instance_name(os.path.basename(os.path.dirname(os.path.dirname(root))))  # Instanzname kürzen
            objectives_type = os.path.basename(os.path.dirname(root)).replace("_", " ")  # Unterstriche entfernen
            strategy = os.path.basename(os.path.dirname(file_path)).replace("_tolerance", " tolerance").replace("hierarchical tolerance", "tolerance")  # Umbenennung
            
            # Mapping der Objectives mit den Namen aus der anderen Tabelle
            result_dict = {
                "Instance": instance,
                "Number of Objectives": objectives_type,
                "Method": strategy,
                "Runtime": round(data.get("RechenzeitInSekunden", None)),  # Runtime auf ganze Zahl runden
                "Construction Fulfillment": data.get("Baustellenfertig", None),
                "Driver Violation": data.get("NichtregulaereFahrer", None),
                "Commute Distance": round(data.get("ArbeitswegGesamt", None), 1),  # Commute Distance auf 1 Nachkommastelle kürzen
                "Transport Distance": round(data.get("TransportdistanzGesamt", None), 1),  # Transport Distance auf 1 Nachkommastelle kürzen
                "Machine Count": data.get("MaschinenGenutzt", None),
                "Worker Count": data.get("ArbeiterGenutzt", None),
            }
            
            # Hinzufügen zur Liste
            data_objectives.append(result_dict)

# Daten in DataFrame umwandeln
df_objectives = pd.DataFrame(data_objectives)

# Strategiereihenfolge hinzufügen
df_objectives["Strategy_Order"] = df_objectives["Method"].map(strategy_order)

# Sortieren: Erst nach Instanz, dann nach Number of Objectives und Method
df_objectives["Instance_Sort"] = df_objectives["Instance"].map(sort_instances)
df_objectives = df_objectives.sort_values(by=["Instance_Sort", "Number of Objectives", "Strategy_Order"]).drop(columns=["Instance_Sort", "Strategy_Order"]).reset_index(drop=True)

# Formatierung: Instance nur einmal anzeigen
df_objectives["Instance"] = df_objectives["Instance"].mask(df_objectives["Instance"].duplicated(), "")

# Formatierung: Number of Objectives pro Gruppe (pro Instanz und Number of Objectives nur einmal anzeigen)
def format_number_of_objectives(df):
    formatted = []
    last_instance = ""
    last_objectives_type = ""
    
    for idx, row in df.iterrows():
        if row["Instance"] != "" or row["Number of Objectives"] != last_objectives_type:
            formatted.append(row["Number of Objectives"])
            last_objectives_type = row["Number of Objectives"]
        else:
            formatted.append("")
    df["Number of Objectives"] = formatted
    return df

df_objectives = format_number_of_objectives(df_objectives)

# Zielreihenfolge der Spalten
desired_order = [
    "Instance",
    "Number of Objectives",
    "Method",
    "Runtime",
    "Construction Fulfillment",
    "Driver Violation",
    "Commute Distance",
    "Transport Distance",
    "Machine Count",
    "Worker Count",
]

# Spalten sortieren
df_objectives = df_objectives[desired_order]

# Werte auf die gerundeten Nachkommastellen beschränken
df_objectives["Runtime"] = df_objectives["Runtime"].astype(int)  # Runtime als ganze Zahl
df_objectives["Commute Distance"] = df_objectives["Commute Distance"].map(lambda x: f"{x:.1f}" if pd.notnull(x) else "")
df_objectives["Transport Distance"] = df_objectives["Transport Distance"].map(lambda x: f"{x:.1f}" if pd.notnull(x) else "")

# Styler-Funktion zum Hinzufügen von dickeren Linien
def style_table(df):
    def highlight_rows(row):
        style = pd.Series("", index=df.columns)
        if row["Instance"] != "":
            style[:] = "border-top: 3px solid black;"  # Dicke Linie über die gesamte Instanz
        if row["Number of Objectives"] != "" and row["Instance"] == "":
            style[1:] = "border-top: 2px solid black;"  # Dicke Linie ab der zweiten Spalte für Number of Objectives
        return style

    return df.style.apply(highlight_rows, axis=1)

# Tabelle mit Stil anzeigen
styled_df = style_table(df_objectives)
styled_df

,Instance,Number of Objectives,Method,Runtime,Construction Fulfillment,Driver Violation,Commute Distance,Transport Distance,Machine Count,Worker Count
0,a3_o80,3 Objectives,costs,8,3,42,3620.9,1365.6,2,8
1,,,weighted,19,3,38,3844.7,1447.4,2,8
2,,,hierarchical,45,3,38,3844.7,1447.4,2,8
3,,,tolerance,38,3,46,3473.5,1701.1,2,8
4,,6 Objectives,costs,3081,3,40,4635.6,70.6,2,6
5,,,weighted,809,2,8,2697.7,29.7,2,5
6,,,hierarchical,1297,3,38,3844.7,917.4,2,8
7,,,tolerance,273,3,46,3811.4,99.2,2,8
8,a5_o96,3 Objectives,costs,16,5,14,5148.0,191.7,7,10
9,,,weighted,46,5,1,6109.3,191.7,7,9
